In [5]:
import os
os.getcwd()

'/Users/subhadeepdutta/Documents/GitHub/algorithmic-trading/notebooks/data_team'

In [9]:
import pandas as pd
df = pd.read_parquet("../../data/clean/clean_prices_50_assets.parquet")
df.head()

Price       Adj Close                                                \
Ticker           AAPL        AMD       AMZN     ASML.AS        BABA   
Date                                                                  
2018-07-26  45.843929  18.350000  90.400002  172.674500  184.411774   
2018-07-27  45.081478  18.940001  90.863503  173.327164  179.891235   
2018-07-30  44.828903  19.420000  88.960999  171.648865  175.522629   
2018-07-31  44.918610  18.330000  88.872002  171.089462  177.811386   
2018-08-01  47.564762  18.480000  89.858498  171.322525  175.949997   

Price                                                                ...  \
Ticker            BAC       BRK-B        COP        COST        CVX  ...   
Date                                                                 ...   
2018-07-26  25.839930  197.460007  56.158089  199.493530  89.403816  ...   
2018-07-27  25.940151  197.949997  55.884327  197.631027  90.860832  ...   
2018-07-30  26.148943  199.089996  56.525703  197.145172  92.202438  ...   
2018-07-31  25.789820  197.869995  56.447475  196.785248  91.077217  ...   
2018-08-01  26.098829  197.850006  56.165901  195.363632  90.442497  ...   

Price          Volume                                                  \
Ticker         SIE.DE         SPY         TLT         TSLA        TSM   
Date                                                                    
2018-07-26  2899906.0  57919500.0   6821100.0   69457500.0  8476600.0   
2018-07-27  2090429.0  76768700.0   4542000.0   85549500.0  9468500.0   
2018-07-30  1224903.0  63742500.0   7111200.0  102211500.0  5174300.0   
2018-07-31  2013498.0  68570500.0   6797400.0   76153500.0  4908300.0   
2018-08-01  2461195.0  53853300.0  13312600.0  151941000.0  6115000.0   

Price                                                                 
Ticker            UNH           V    VOW3.DE         WMT         XOM  
Date                                                                  
2018-07-26  2021300.0  10520800.0  2197625.0  19644900.0  13210100.0  
2018-07-27  1656800.0   7014900.0  1116046.0  14038200.0  18220800.0  
2018-07-30  1681500.0  12812400.0   809150.0  18032100.0  11282800.0  
2018-07-31  2355900.0   7683300.0  1270584.0  20124300.0  11716800.0  
2018-08-01  3006700.0   7087700.0  2761582.0  15150000.0   8936300.0  

[5 rows x 300 columns]

In [14]:
adj_close = df["Adj Close"].copy()
adj_close.head()

Ticker,AAPL,AMD,AMZN,ASML.AS,BABA,BAC,BRK-B,COP,COST,CVX,...,SIE.DE,SPY,TLT,TSLA,TSM,UNH,V,VOW3.DE,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2018-07-26,45.843929,18.350000,90.400002,172.674500,184.411774,25.839930,197.460007,56.158089,199.493530,89.403816,...,87.426346,252.196243,96.831596,20.443333,34.915237,227.677277,135.267166,91.138329,26.194304,59.618633
2018-07-27,45.081478,18.940001,90.863503,173.327164,179.891235,25.940151,197.949997,55.884327,197.631027,90.860832,...,87.716843,250.487198,96.993996,19.812000,35.454102,226.747101,133.568039,90.789886,26.164614,57.976734
2018-07-30,44.828903,19.420000,88.960999,171.648865,175.522629,26.148943,199.089996,56.525703,197.145172,92.202438,...,87.368240,249.178772,96.661110,19.344667,35.146175,224.877808,129.552719,90.994171,26.387280,57.849339
2018-07-31,44.918610,18.330000,88.872002,171.089462,177.811386,25.789820,197.869995,56.447475,196.785248,91.077217,...,87.702324,250.407150,97.188850,19.875999,35.248821,224.328568,129.799545,91.450745,26.491190,57.686565
2018-08-01,47.564762,18.480000,89.858498,171.322525,175.949997,26.098829,197.850006,56.165901,195.363632,90.442497,...,87.019630,249.988754,96.397072,20.056000,35.676491,224.461502,131.232910,88.074379,26.197268,56.893898


In [25]:
adj_close_long = (
    adj_close
    .reset_index()
    .melt(
        id_vars= "Date",
        var_name="ticker",
        value_name="adj_close"
    )
)
adj_close_long.head()


,Date,ticker,adj_close
0,2018-07-26,AAPL,45.843929
1,2018-07-27,AAPL,45.081478
2,2018-07-30,AAPL,44.828903
3,2018-07-31,AAPL,44.918610
4,2018-08-01,AAPL,47.564762


In [26]:
adj_close_long = adj_close_long.sort_values(["ticker", "Date"])

In [29]:
import numpy as np
df = adj_close_long.copy()

df["daily_return"] = df.groupby("ticker")["adj_close"].pct_change()

df["return_12m"] = df.groupby("ticker")["adj_close"].pct_change(252)

df["ma_20"] = df.groupby("ticker")["adj_close"].transform(lambda s: s.rolling(20).mean())
df["ma_50"] = df.groupby("ticker")["adj_close"].transform(lambda s: s.rolling(50).mean())

df["vol_20"] = df.groupby("ticker")["daily_return"].transform(lambda s: s.rolling(20).std())

df["z_20"] = (df["adj_close"]- df["ma_20"])/ df["vol_20"]

In [67]:
features_50 = df.dropna(
    subset= ["daily_return", "return_12m", "ma_20", "ma_50", "vol_20", "z_20"]
).copy()

features_50.to_parquet("../../data/features_50.parquet", index= False)

In [70]:
features_50 = features_50.rename(columns={"Date": "date"})

In [78]:
features_50.columns

Index(['date', 'ticker', 'adj_close', 'daily_return', 'return_12m', 'ma_20',
       'ma_50', 'vol_20', 'z_20'],
      dtype='object')

In [79]:
features_50.head()

,date,ticker,adj_close,daily_return,return_12m,ma_20,ma_50,vol_20,z_20
252,2019-08-14,AAPL,48.772968,-0.029765,0.063892,49.256848,47.849188,0.019210,-25.189093
253,2019-08-15,AAPL,48.529999,-0.004982,0.076495,49.246740,47.980690,0.019201,-37.329049
254,2019-08-16,AAPL,49.675056,0.023595,0.108103,49.266207,48.143576,0.019769,20.681136
255,2019-08-19,AAPL,50.601212,0.018644,0.126509,49.368767,48.294597,0.019809,62.217898
256,2019-08-20,AAPL,50.603611,0.000047,0.063889,49.415969,48.431768,0.019208,61.831380


In [80]:
os.path.exists("../../data/features.parquet")

True